# Notebook de réplication d’un autocall par vanilles

## Objectif

L’idée de ce notebook est de construire une base propre pour étudier la réplication statique d’un autocall par un panier fini de vanilles.

On veut ici :

- définir clairement le produit cible ;
- poser un modèle de pricing de référence ;
- construire un univers de vanilles ;
- tester un benchmark naïf ;
- rendre le tout plus lisible et plus simple à déboguer.


In [ ]:
from __future__ import annotations

from typing import Any, Dict, Optional
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from IPython.display import display


import yfinance as yf

In [ ]:
np.set_printoptions(linewidth=10000)

## 2. Principe général

On part d’un autocall simple, puis on cherche à l’approximer par une combinaison linéaire de 100 à 200 vanilles observables sur le marché.

La logique du notebook est la suivante :

1. construire le payoff cible de l’autocall ;
2. construire un univers de vanilles ;
3. price les deux dans un même cadre ;
4. résoudre un problème de fit direct ;
5. mesurer la qualité de l’approximation ;
6. identifier les limites de cette baseline.

Cette baseline servira de point de comparaison pour une méthode plus intelligente fondée sur la structure du produit.

---

## 3. Produit cible

On commence par un autocall simple, à une seule sous-jacente, avec :

- plusieurs dates de constatation ;
- un coupon fixe ;
- un niveau d’autocall ;
- une barrière down éventuelle ;
- un payoff terminal simple en cas de non-rappel.

Il est important de ne pas introduire trop de complexité dès le départ.  
Le produit doit être assez riche pour être représentatif, mais assez simple pour être codé, testé et debuggué rapidement.

In [ ]:
import importlib
import products_core
importlib.reload(products_core)

from products_core import AutocallProduct, VanillaProduct, EuropeanCall, EuropeanPut, BinaryCall, BinaryPut, Cash, build_vanilla_universe, vanilla_universe_to_dataframe

## Tests de cohérence du payoff de l’autocall

On construit ici trois trajectoires de prix artificielles pour vérifier que la fonction `compute_autocall_payoff` réagit correctement dans les trois situations clés :

- **Cas 1** : le produit est appelé au premier coupon.
- **Cas 2** : le produit n’est jamais appelé et la barrière n’est jamais touchée.
- **Cas 3** : le produit n’est pas appelé, mais la barrière est touchée pendant la vie du produit.

In [ ]:
dates = pd.bdate_range("2025-01-01", periods=280)    # 1 an de jours ouvrés
prices = pd.Series(100.0, index=dates)

# Cas 1 : autocall au premier coupon
prices1 = prices.copy()
prices1.loc[dates[63:]] = 101.0  # autour du 3e mois

# Cas 2 : jamais appelé, pas de barrière
prices2 = prices.copy()             # spot initial constant à 99% du nominal
prices2.loc[:] = 99.0

# Cas 3 : barrière touchée = 70
prices3 = prices.copy()
prices3.loc[dates[50:80]] = 55.0
prices3.loc[dates[80:]] = 99.0

prod = AutocallProduct()

res1 = prod.compute_autocall_payoff(prices1, start_date=dates[0])
res2 = prod.compute_autocall_payoff(prices2, start_date=dates[0])
res3 = prod.compute_autocall_payoff(prices3, start_date=dates[0])

print("Called : ", res1["called"], "/ Call Date : ", res1["call_date"], "/ Undiscounted Payoff : ", res1["undiscounted_payoff"])
print("Called : ", res2["called"], "/ Call Date : ", res2["call_date"], "/ Barrier Hit : ", res2["barrier_hit"], "/ Undiscounted Payoff : ", res2["undiscounted_payoff"])
print("Called : ", res3["called"], "/ Call Date : ", res3["call_date"], "/ Barrier Hit : ", res3["barrier_hit"], "/ Undiscounted Payoff : ", res3["undiscounted_payoff"])

## 4. Univers de vanilles

On construit ensuite une base de vanilles couvrant :

- plusieurs maturités ;
- plusieurs strikes autour du spot ;
- des calls et des puts ;
- idéalement des instruments proches des dates de call de l’autocall.

L’idée est de disposer d’un dictionnaire d’instruments suffisamment riche pour que le fit direct ait une chance raisonnable de fonctionner.

Le point critique ici est la diversité du panier.  
Un panier trop concentré autour de l’ATM donne un fit fragile ; un panier trop large peut devenir instable numériquement sans régularisation.

---

## 5. Modèle de pricing de référence

Pour la première version du notebook, on choisit un cadre de pricing simple et contrôlé.

Le plus naturel est de commencer avec :

- Black-Scholes comme modèle de base ;
- éventuellement une surface implicite en second temps ;
- puis plus tard un modèle plus riche si nécessaire.

Le but est de séparer clairement :

- l’erreur liée à la réplication ;
- l’erreur liée au modèle de prix ;
- l’erreur liée à la discrétisation.

Tant que ces trois sources d’erreur ne sont pas séparées, les résultats seront difficiles à interpréter.

In [ ]:
from products_core import norm_cdf, norm_pdf, bs_d1_d2, _year_fraction

## Test des produits européens et des parités Black-Scholes

On fixe ici un jeu de paramètres simple afin de tester les produits de base :

- un call européen,
- un put européen,
- un call binaire,
- un put binaire.

L’objectif est de vérifier que :
- les prix Black-Scholes sont cohérents,
- la parité put-call est bien respectée,
- la parité entre les deux options binaires est bien vérifiée,
- les payoffs terminaux correspondent aux définitions théoriques.

In [ ]:
# ------------------------------------------------------------
# Paramètres de test
# ------------------------------------------------------------
valuation_date = pd.Timestamp("2026-06-20")
maturity = pd.Timestamp("2027-06-20")

spot = 100.0
strike = 100.0
rate = 0.03
dividend_yield = 0.01
vol = 0.20              
# On suppose que la volatilité est constante pour simplifier l'exemple, et unique pour toutes les options. 
# Dans la réalité, on utiliserait une surface de volatilité.

# ------------------------------------------------------------
# Création des produits
# ------------------------------------------------------------
call = EuropeanCall(name="Call_100",        strike=strike,maturity_date=maturity,notional=1.0)
put = EuropeanPut(  name="Put_100",         strike=strike,maturity_date=maturity,notional=1.0)
bcall = BinaryCall( name="BinaryCall_100",  strike=strike,maturity_date=maturity,notional=1.0)
bput = BinaryPut(   name="BinaryPut_100",   strike=strike,maturity_date=maturity,notional=1.0)

# ------------------------------------------------------------
# Prix Black-Scholes
# ------------------------------------------------------------
c = call.   price_bs(spot, valuation_date, rate, dividend_yield, vol)
p = put.    price_bs(spot, valuation_date, rate, dividend_yield, vol)
bc = bcall. price_bs(spot, valuation_date, rate, dividend_yield, vol)
bp = bput.  price_bs(spot, valuation_date, rate, dividend_yield, vol)

print("Call      :", c)
print("Put       :", p)
print("Binary call:", bc)
print("Binary put :", bp)

# ------------------------------------------------------------
# Tests de cohérence
# ------------------------------------------------------------
T = _year_fraction(valuation_date, maturity)

# Parité put-call : C - P = S0 * exp(-q*T) - K * exp(-r*T)
# Parité binaire : BC + BP = exp(-r*T)
lhs = c - p
rhs = spot * math.exp(-dividend_yield * T) - strike * math.exp(-rate * T)

print("\nPut-call parity: C - P = S0 * exp(-q*T) - K * exp(-r*T)")
print("LHS =", lhs)
print("RHS =", rhs)
print("Erreur absolue =", abs(lhs - rhs))

print("\nBinary parity: BC + BP = e^{-rT}")
print("BC + BP =", bc + bp)
print("e^{-rT}  =", math.exp(-rate * T))
print("Erreur absolue =", abs((bc + bp) - math.exp(-rate * T)))

# ------------------------------------------------------------
# Payoff terminal simple
# ------------------------------------------------------------
spot_T = 112.0
print("\nPayoffs à S_T = 112:")
print("Call payoff :", call.payoff(spot_T))
print("Put payoff  :", put.payoff(spot_T))
print("BC payoff   :", bcall.payoff(spot_T))
print("BP payoff   :", bput.payoff(spot_T))

## Simulation Black-Scholes d'une trajectoire de prix

Cette fonction génère une trajectoire quotidienne du sous-jacent sous le modèle Black-Scholes.  
Elle servira ensuite à simuler des scénarios de marché pour le pricing Monte Carlo de l'autocall.

In [ ]:
import monte_carlo
importlib.reload(monte_carlo)

from monte_carlo import simulate_gbm_path

## Pricing Monte Carlo de l'autocall

On simule un grand nombre de trajectoires du sous-jacent, puis on calcule le payoff de l'autocall sur chaque trajectoire.  
Le prix final est obtenu en moyennant les cashflows actualisés.

In [ ]:
from monte_carlo import price_autocall_bs_mc

## Construction de la grille de travail

On construit ici deux axes de calcul :
- une liste de dates pertinentes,
- une grille de spots autour du spot initial.

Cette grille sert à étudier la valeur de l'autocall en fonction du temps et du niveau du sous-jacent.

In [ ]:
import benchmark_general
importlib.reload(benchmark_general)
from benchmark_general import build_autocall_training_grid_axes, build_naive_vanilla_basis

## Surface de prix sur la grille

On calcule le prix de l'autocall pour chaque couple (date, spot) de la grille afin d'obtenir une matrice de prix.

In [ ]:
import target_price
importlib.reload(target_price)

from target_price import price_autocall_on_tensor_bs_mc

## Prix de l'autocall sur une grille de dates et de spots

On instancie ici un autocall simple, puis on calcule :

1. un prix Monte Carlo sur un scénario de référence ;
2. une matrice de prix sur une grille de travail composée :
   - de plusieurs dates pertinentes,
   - de plusieurs niveaux de spot autour de `spot0`.

Le tableau affiché représente donc une surface de prix de l'autocall :
- les **lignes** correspondent aux dates,
- les **colonnes** correspondent aux spots,
- chaque case contient le prix estimé du produit pour ce couple (date, spot).

Cela permet de visualiser comment la valeur de l'autocall varie à la fois avec le temps et avec le niveau du sous-jacent.

In [ ]:
prod = AutocallProduct(notional=100.0,autocall_strike=1.00,coupon_rate=0.02,down_barrier=0.70,observation_months=[3, 6, 9, 12],)

valuation_date = pd.Timestamp("2026-06-20")
maturity_date = pd.Timestamp("2027-06-20")
spot0 = 100.0
rate = 0.03
dividend_yield = 0.01
vol = 0.20

res = price_autocall_bs_mc(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,n_paths=2000)
print(res["actualized_price"], res["std_error"])

In [ ]:
dates, spots = build_autocall_training_grid_axes(spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,n_spots=15,spot_min_mult=0.6,spot_max_mult=1.4,include_call_dates=True,product=prod)

price_tensor = price_autocall_on_tensor_bs_mc(product=prod,dates=dates,spots=spots,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,n_paths=200,seed=42)

df_view = pd.DataFrame(price_tensor, index=[d.strftime("%Y-%m-%d") for d in dates], columns=np.round(spots, 2))          # tableau des prix simulés de l'autocall sur la grille (dates en lignes, spots en colonnes)       
display(df_view)
print("Prix de l'autocall : lignes = dates, colonnes = spots")


#Les 4 lignes viennent des dates d’observation : typiquement 3, 6, 9, 12 mois si observation_months = [3, 6, 9, 12].
#Les 15 colonnes viennent de n_spots=15, donc tu as pris 15 valeurs de spot entre 0.6 * spot0 et 1.4 * spot0.
#La dernière ligne entièrement à 100 suggère qu’à cette date, le payoff est identique quel que soit le spot dans ta grille, ce qui est cohérent avec un produit arrivé à une situation où le remboursement est fixé par le contrat.

## 6. Formulation du benchmark naïf : réplication par un portefeuille de vanilles

Le benchmark consiste à approximer la valeur de l’autocall par une combinaison linéaire de vanilles :

$$
V_{\text{autocall}}(d,s) \approx \sum_{i=1}^N w_i V_i(d,s)
$$

où :

- $V_{\text{autocall}}$ est la valeur du produit cible ;
- $V_i$ est la valeur de la vanille $i$ ;
- $w_i$ est le poids associé.
- d = une date de valorisation
- s = un niveau de spot 

Ce benchmark doit être traité comme une approximation statique simple, pas comme une réplication exacte.
L'idée est volontairement naïve :
- on construit une base de produits simples (calls / puts, éventuellement en grande quantité),
- on ajuste leurs poids pour reproduire au mieux la surface de prix de l'autocall,
- on compare ensuite la qualité de la réplication obtenue.

Ce benchmark sert de point de comparaison simple avant d'aller vers des méthodes plus sophistiquées de calibration ou d'approximation.

## Benchmark naïf par étapes

On construit ici une réplication simple de l'autocall à l'aide d'options vanilles européennes, en procédant par niveaux de complexité.

La démarche est la suivante :
1. commencer avec un portefeuille composé uniquement de calls ;
2. enrichir ensuite la base avec des calls et des puts ;
3. ajouter enfin les options binaires pour vérifier si elles améliorent sensiblement la qualité de réplication.

L’objectif est de comparer, étape par étape, la capacité de chaque famille de produits à reproduire la surface de prix de l’autocall.

In [ ]:
# ============================================================
# 6. BENCHMARK NAÏF :     Calls -> Calls/Puts -> Calls/Puts/Binaires
# ============================================================

# ------------------------------------------------------------
# Pricing d'un panier sur une grille (dates x spots)
# ------------------------------------------------------------
from target_price import price_vanilla_basis_on_grid

In [ ]:
# ------------------------------------------------------------
# 3) Fit ridge simple (benchmark naïf)
# ------------------------------------------------------------
from benchmark_general import fit_linear, fit_lstsq


# ------------------------------------------------------------
# 4) Métriques
# ------------------------------------------------------------
from benchmark_general import benchmark_metrics

In [ ]:
# ------------------------------------------------------------
# 5) Pipeline complet
# ------------------------------------------------------------
from target_price import run_naive_benchmark_price

# ------------------------------------------------------------
# 6) Affichage / comparaison
# ------------------------------------------------------------
from benchmark_general import plot_benchmark_surface

In [ ]:
# ============================================================
# EXEMPLES D'UTILISATION
# ============================================================
for penalty in ["l1","l2","huber", "elastic_net", "l1_2"] :
    res_calls =         run_naive_benchmark_price(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="calls",n_spots=15,n_paths=200, penalty=penalty)
    res_calls_puts =    run_naive_benchmark_price(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="calls_puts",n_spots=15,n_paths=200, penalty=penalty)
    res_full =          run_naive_benchmark_price(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="full",n_spots=15,n_paths=200, penalty=penalty)

    print(f"=== CALLS - {penalty} ===")
    print("Train:", res_calls["metrics_train"])
    print("Test :", res_calls["metrics_test"])
    display(res_calls["weights"].head(20))

    print(f"=== CALLS + PUTS - {penalty} ===")
    print("Train:", res_calls_puts["metrics_train"])
    print("Test :", res_calls_puts["metrics_test"])
    display(res_calls_puts["weights"].head(20))

    print(f"=== FULL - {penalty} ===")
    print("Train:", res_full["metrics_train"])
    print("Test :", res_full["metrics_test"])
    display(res_full["weights"].head(20))
    

    
    comparative_table = pd.DataFrame([
    {
            "family": "calls",
            "train_mae": res_calls["metrics_train"]["mae"],
            "train_rmse": res_calls["metrics_train"]["rmse"],
            "train_max_abs_error": res_calls["metrics_train"]["max_abs_error"],
            "train_mean_error": res_calls["metrics_train"]["mean_error"],
            "test_mae": res_calls["metrics_test"].get("mae", np.nan),
            "test_rmse": res_calls["metrics_test"].get("rmse", np.nan),
            "test_max_abs_error": res_calls["metrics_test"].get("max_abs_error", np.nan),
            "test_mean_error": res_calls["metrics_test"].get("mean_error", np.nan),
        },
        {
            "family": "calls_puts",
            "train_mae": res_calls_puts["metrics_train"]["mae"],
            "train_rmse": res_calls_puts["metrics_train"]["rmse"],
            "train_max_abs_error": res_calls_puts["metrics_train"]["max_abs_error"],
            "train_mean_error": res_calls_puts["metrics_train"]["mean_error"],
            "test_mae": res_calls_puts["metrics_test"].get("mae", np.nan),
            "test_rmse": res_calls_puts["metrics_test"].get("rmse", np.nan),
            "test_max_abs_error": res_calls_puts["metrics_test"].get("max_abs_error", np.nan),
            "test_mean_error": res_calls_puts["metrics_test"].get("mean_error", np.nan),
        },
        {
            "family": "full",
            "train_mae": res_full["metrics_train"]["mae"],
            "train_rmse": res_full["metrics_train"]["rmse"],
            "train_max_abs_error": res_full["metrics_train"]["max_abs_error"],
            "train_mean_error": res_full["metrics_train"]["mean_error"],
            "test_mae": res_full["metrics_test"].get("mae", np.nan),
            "test_rmse": res_full["metrics_test"].get("rmse", np.nan),
            "test_max_abs_error": res_full["metrics_test"].get("max_abs_error", np.nan),
            "test_mean_error": res_full["metrics_test"].get("mean_error", np.nan),
        },
    ])
    pd.reset_option("display.float_format")
    display(comparative_table)

    metrics_plot = comparative_table.copy()
    metrics_plot["family"] = ["Calls", "Calls + Puts", "Full"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

    # Train RMSE / Test RMSE
    axes[0].bar(metrics_plot["family"], metrics_plot["train_rmse"], label="Train RMSE", alpha=0.8)
    axes[0].bar(metrics_plot["family"], metrics_plot["test_rmse"], label="Test RMSE", alpha=0.8)
    axes[0].set_title("RMSE comparatif")
    axes[0].set_ylabel("RMSE")
    axes[0].tick_params(axis="x", rotation=20)
    axes[0].legend()

    # Train MAE / Test MAE
    axes[1].bar(metrics_plot["family"], metrics_plot["train_mae"], label="Train MAE", alpha=0.8)
    axes[1].bar(metrics_plot["family"], metrics_plot["test_mae"], label="Test MAE", alpha=0.8)
    axes[1].set_title("MAE comparatif")
    axes[1].set_ylabel("MAE")
    axes[1].tick_params(axis="x", rotation=20)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    plot_benchmark_surface(res_calls["dates"],res_calls["spots"],res_calls["target_tensor"],res_calls["fitted_tensor"], title_prefix="[Calls] ")
    plot_benchmark_surface(res_calls_puts["dates"],res_calls_puts["spots"],res_calls_puts["target_tensor"],res_calls_puts["fitted_tensor"], title_prefix="[Calls + PUTS] ")
    plot_benchmark_surface(res_full["dates"],res_full["spots"],res_full["target_tensor"],res_full["fitted_tensor"], title_prefix="[Full] ")

### Pourquoi utiliser `price_bs` ici plutôt que `payoff` ?

Dans cette fonction, on ne cherche pas le payoff d’une vanille à maturité, mais **l’espérance de son payoff brut au point de grille** \((t, S_t)\).

- `payoff(S_T)` donne la valeur finale de l’option **si l’on connaît le spot terminal** \(S_T\).
- `price_bs(...)` donne la **valeur actualisée** aujourd’hui :

$$
\text{price} = e^{-r\tau}\,\mathbb{E}^Q[\text{payoff}]
$$

d’où :

$$
\mathbb{E}^Q[\text{payoff}] = \text{price} \cdot e^{r\tau}
$$

Donc :

- si \(\tau \le 0\), on est à maturité ou après, et on peut utiliser directement `payoff(s)`;
- si \(\tau > 0\), `payoff(s)` ne suffit pas, car il faudrait le **spot terminal** \(S_T\), pas le spot courant \(s\).

C’est pour cela qu’on utilise `price_bs(...)` puis qu’on reconvertit en **espérance brute du payoff** avec `* exp(r * tau)`.  
C’est une façon propre et exacte de construire la matrice de régression pour les vanilles européennes.

In [ ]:
# ============================================================
# BENCHMARK PAYOFF : autocall vs portefeuille de vanilles
# ============================================================

import benchmark_mean_payoff
importlib.reload(benchmark_mean_payoff)

# ------------------------------------------------------------
# 0) Helper : moyenne MC du payoff brut d'un autocall
# ------------------------------------------------------------
from benchmark_mean_payoff import mean_undiscounted_payoff_mc

# ------------------------------------------------------------
# 1) Cible : espérance du payoff brut de l'autocall sur une grille
# ------------------------------------------------------------
from benchmark_mean_payoff import autocall_meanpayoff_on_grid_mc

# ------------------------------------------------------------
# 2) Réplique : espérance de payoff brut du portefeuille de vanilles
# ------------------------------------------------------------
from benchmark_mean_payoff import vanilla_basis_expected_payoff_on_grid

# ------------------------------------------------------------
# 3) Pipeline complet : fit sur payoff brut
# ------------------------------------------------------------
from benchmark_mean_payoff import run_naive_benchmark_payoff



In [ ]:
# ------------------------------------------------------------
# 5) Exécution
# ------------------------------------------------------------
res_payoff_calls =      run_naive_benchmark_payoff(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="calls",n_spots=15,n_paths=200,penalty="l2",alpha=1e-6)
res_payoff_calls_puts = run_naive_benchmark_payoff(product=prod,spot0=spot0,valuation_date=valuation_date, maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="calls_puts",n_spots= 15,n_paths=200,penalty="l2",alpha=1e-6)
res_payoff_full =       run_naive_benchmark_payoff(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="full",n_spots=15,n_paths=200,penalty="l2",alpha=1e-6)

print("=== CALLS ===")
print("Train:", res_payoff_calls["metrics_train"])
print("Test :", res_payoff_calls["metrics_test"])
display(res_payoff_calls["weights"].head(10))

print("=== CALLS + PUTS ===")
print("Train:", res_payoff_calls_puts["metrics_train"])
print("Test :", res_payoff_calls_puts["metrics_test"])
display(res_payoff_calls_puts["weights"].head(10))

print("=== FULL ===")
print("Train:", res_payoff_full["metrics_train"])
print("Test :", res_payoff_full["metrics_test"])
display(res_payoff_full["weights"].head(10))

plot_benchmark_surface(res_payoff_calls["dates"],res_payoff_calls["spots"],res_payoff_calls["target_tensor"],res_payoff_calls["fitted_tensor"],title_prefix="[Calls] ")
plot_benchmark_surface(res_payoff_calls_puts["dates"],res_payoff_calls_puts["spots"],res_payoff_calls_puts["target_tensor"],res_payoff_calls_puts["fitted_tensor"],title_prefix="[Calls + PUTS] ")
plot_benchmark_surface(res_payoff_full["dates"],res_payoff_full["spots"],res_payoff_full["target_tensor"],res_payoff_full["fitted_tensor"],title_prefix="[Full] ")

In [ ]:
comparative_payoff_table = pd.DataFrame([
    {
        "family": "calls",
        "train_mae": res_payoff_calls["metrics_train"]["mae"],
        "train_rmse": res_payoff_calls["metrics_train"]["rmse"],
        "train_max_abs_error": res_payoff_calls["metrics_train"]["max_abs_error"],
        "train_mean_error": res_payoff_calls["metrics_train"]["mean_error"],
        "test_mae": res_payoff_calls["metrics_test"].get("mae", np.nan),
        "test_rmse": res_payoff_calls["metrics_test"].get("rmse", np.nan),
        "test_max_abs_error": res_payoff_calls["metrics_test"].get("max_abs_error", np.nan),
        "test_mean_error": res_payoff_calls["metrics_test"].get("mean_error", np.nan),
    },
    {
        "family": "calls_puts",
        "train_mae": res_payoff_calls_puts["metrics_train"]["mae"],
        "train_rmse": res_payoff_calls_puts["metrics_train"]["rmse"],
        "train_max_abs_error": res_payoff_calls_puts["metrics_train"]["max_abs_error"],
        "train_mean_error": res_payoff_calls_puts["metrics_train"]["mean_error"],
        "test_mae": res_payoff_calls_puts["metrics_test"].get("mae", np.nan),
        "test_rmse": res_payoff_calls_puts["metrics_test"].get("rmse", np.nan),
        "test_max_abs_error": res_payoff_calls_puts["metrics_test"].get("max_abs_error", np.nan),
        "test_mean_error": res_payoff_calls_puts["metrics_test"].get("mean_error", np.nan),
    },
    {
        "family": "full",
        "train_mae": res_payoff_full["metrics_train"]["mae"],
        "train_rmse": res_payoff_full["metrics_train"]["rmse"],
        "train_max_abs_error": res_payoff_full["metrics_train"]["max_abs_error"],
        "train_mean_error": res_payoff_full["metrics_train"]["mean_error"],
        "test_mae": res_payoff_full["metrics_test"].get("mae", np.nan),
        "test_rmse": res_payoff_full["metrics_test"].get("rmse", np.nan),
        "test_max_abs_error": res_payoff_full["metrics_test"].get("max_abs_error", np.nan),
        "test_mean_error": res_payoff_full["metrics_test"].get("mean_error", np.nan),
    },
])

display(comparative_payoff_table)

metrics_plot = comparative_payoff_table.copy()
metrics_plot["family"] = ["Calls", "Calls + Puts", "Full"]

x = np.arange(len(metrics_plot))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].bar(x - width/2, metrics_plot["train_rmse"], width, label="Train RMSE", alpha=0.8)
axes[0].bar(x + width/2, metrics_plot["test_rmse"], width, label="Test RMSE", alpha=0.8)
axes[0].set_title("RMSE comparatif")
axes[0].set_ylabel("RMSE")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_plot["family"], rotation=20)
axes[0].legend()

axes[1].bar(x - width/2, metrics_plot["train_mae"], width, label="Train MAE", alpha=0.8)
axes[1].bar(x + width/2, metrics_plot["test_mae"], width, label="Test MAE", alpha=0.8)
axes[1].set_title("MAE comparatif")
axes[1].set_ylabel("MAE")
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_plot["family"], rotation=20)
axes[1].legend()

plt.tight_layout()
plt.show()

plot_benchmark_surface(res_payoff_calls["dates"],res_payoff_calls["spots"],res_payoff_calls["target_tensor"],res_payoff_calls["fitted_tensor"],title_prefix="[Calls] ")
plot_benchmark_surface(res_payoff_calls_puts["dates"],res_payoff_calls_puts["spots"],res_payoff_calls_puts["target_tensor"],res_payoff_calls_puts["fitted_tensor"],title_prefix="[Calls + PUTS] ")
plot_benchmark_surface(res_payoff_full["dates"],res_payoff_full["spots"],res_payoff_full["target_tensor"],res_payoff_full["fitted_tensor"],title_prefix="[Full] ")

In [ ]:
# ============================================================
# BENCHMARK PAYOFF PATHWISE : mêmes trajectoires pour tout
# ============================================================

import benchmark_payoff
importlib.reload(benchmark_payoff)

# ------------------------------------------------------------
# 0) Payoff actualisé de l'autocall sur une trajectoire donnée
# ------------------------------------------------------------
from monte_carlo import autocall_discounted_payoff_from_path

# ------------------------------------------------------------
# 1) Payoff actualisé d'une vanille sur une trajectoire donnée
# ------------------------------------------------------------
from benchmark_payoff import vanilla_discounted_payoff_from_path

# ------------------------------------------------------------
# 2) Construction du dataset pathwise
# ------------------------------------------------------------
from benchmark_payoff import build_pathwise_payoff_dataset

# ------------------------------------------------------------
# 3) Pipeline complet : benchmark payoff pathwise
# ------------------------------------------------------------
from benchmark_payoff import run_naive_benchmark_payoff_pathwise

# ------------------------------------------------------------
# 4) Affichage : version lisible et interprétable
# ------------------------------------------------------------
from benchmark_payoff import plot_pathwise_benchmark

In [ ]:
# ------------------------------------------------------------
# 5) Exécution
# ------------------------------------------------------------

res_path_calls = run_naive_benchmark_payoff_pathwise(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="calls",n_paths=300,penalty="l2",alpha=1e-6)
res_path_calls_puts = run_naive_benchmark_payoff_pathwise(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="calls_puts",n_paths=300,penalty="l2",alpha=1e-6)
res_path_full = run_naive_benchmark_payoff_pathwise(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family="full",n_paths=300,penalty="l2",alpha=1e-6)

print("Train:", res_path_calls["metrics_train"])
print("Test :", res_path_calls["metrics_test"])
display(res_path_calls["weights"].head(10))

print("Train:", res_path_calls_puts["metrics_train"])
print("Test :", res_path_calls_puts["metrics_test"])
display(res_path_calls_puts["weights"].head(10))

print("Train:", res_path_full["metrics_train"])
print("Test :", res_path_full["metrics_test"])
display(res_path_full["weights"].head(10))

plot_pathwise_benchmark(res_path_calls["y"],res_path_calls["replica"],title_prefix="[Pathwise / Calls] ")
plot_pathwise_benchmark(res_path_calls_puts["y"],res_path_calls_puts["replica"],title_prefix="[Pathwise / Calls + Puts] ")
plot_pathwise_benchmark(res_path_full["y"],res_path_full["replica"],title_prefix="[Pathwise / Full] ")


In [ ]:
# ============================================================
# BENCHMARK COURBE PATHWISE : cashflows de l'autocall et du réplica le long d'une trajectoire
# ============================================================-

import benchmark_static_payoff_pathwise
importlib.reload(benchmark_static_payoff_pathwise)

# ------------------------------------------------------------
# Fonction utilitaire : date réellement disponible sur le path pour une date contractuelle
# ------------------------------------------------------------
from benchmark_static_payoff_pathwise import effective_date_on_path

# ------------------------------------------------------------
# 0) Courbe cible : cashflows actualisés de l'autocall sur un path
# ------------------------------------------------------------
from benchmark_static_payoff_pathwise import autocall_value_curve_on_path

# ------------------------------------------------------------
# 1) Courbe réplique : cashflows actualisés du portefeuille de vanilles sur un path
# ------------------------------------------------------------
from benchmark_static_payoff_pathwise import vanilla_portfolio_value_curve_on_path

# ------------------------------------------------------------
# 1.1) Affichage du panier retenu sur un path
# ------------------------------------------------------------

from benchmark_static_payoff_pathwise import replication_basket_on_path

# ------------------------------------------------------------
# 2) Dataset pathwise temporel :     chaque observation = (path, date de cashflow)
#     """Construit une liste de trajectoires.
#    Pour chaque trajectoire, on stocke :
#    - le path simulé ;
#    - les cashflows actualisés de l'autocall par date ici on note y_curve ;
#    - les cashflows actualisés du réplica par date ici on note X_curve ;
#    - les dates de cashflows observées (y_curve.index)"""
# ------------------------------------------------------------

from benchmark_static_payoff_pathwise  import build_timewise_curve_dataset

# ------------------------------------------------------------
# 3) Pipeline complet : fit sur les cashflows pathwise temporels
# ------------------------------------------------------------

from benchmark_static_payoff_pathwise import run_naive_benchmark_curve

# ------------------------------------------------------------
# 4) Plot lisible :   cashflows ponctuels et cashflows cumulés
# ------------------------------------------------------------

from benchmark_static_payoff_pathwise import plot_timewise_curve_benchmark


In [ ]:
test_path = simulate_gbm_path(spot0=spot0,start_date=pd.Timestamp(valuation_date),end_date=pd.Timestamp(maturity_date),rate=rate,dividend_yield=dividend_yield,vol=vol,seed=10500)

for penalty in ["l2", "l1", "elastic_net"]:
    for family in ["calls", "calls_puts", "full"]:
        res_curve = run_naive_benchmark_curve(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family=family,n_paths=2000,penalty=penalty,alpha=1e-4)
        target_curve = autocall_value_curve_on_path(product=prod,path=test_path,valuation_date=valuation_date,rate=rate,curve_dates=res_curve["dates"])
        replica_curve = vanilla_portfolio_value_curve_on_path(vanillas=res_curve["vanillas"],weights=res_curve["w"],path=test_path,valuation_date=valuation_date,rate=rate,curve_dates=target_curve.index)

        plot_timewise_curve_benchmark(target_curve.index,target_curve.to_numpy(),replica_curve.to_numpy(),title_prefix=f"[Cashflows / {family} / {penalty}] ")

        basket = replication_basket_on_path(result=res_curve,path=test_path,valuation_date=valuation_date,rate=rate)

        print(f"=== {family.upper()} / {penalty} ===")

        print("Métriques train :")
        display(pd.Series(res_curve["metrics_train"]))

        print("Métriques test :")
        display(pd.Series(res_curve["metrics_test"]))

        print("Métriques test par date :")
        display(res_curve["metrics_by_date"])

        print("Panier de réplication :")
        display(basket.head(10))

In [ ]:
# ============================================================
# BENCHMARK CONDITIONNEL SEMI-STATIQUE : un panier par date et par état observable du produit
# ============================================================

import benchmark_semi_static_payoff_pathwise
importlib.reload(benchmark_semi_static_payoff_pathwise)

# ------------------------------------------------------------
# 0) Construction de la base conditionnelle :  une jambe cash et des vanilles à chaque date d'observation
# ------------------------------------------------------------

from benchmark_semi_static_payoff_pathwise import build_conditional_vanilla_basis

# ------------------------------------------------------------
# 1) États observables du produit le long d'un path : alive_before et barrier_hit
# ------------------------------------------------------------

from benchmark_semi_static_payoff_pathwise  import autocall_state_curve_on_path

# ------------------------------------------------------------
# 2) Dataset conditionnel :   on enrichit le dataset actuel avec les états du produit
# ------------------------------------------------------------

from benchmark_semi_static_payoff_pathwise  import build_conditional_curve_dataset

# ------------------------------------------------------------
# 3) Fit des paniers conditionnels
#    fit_linear normalise déjà les colonnes et renvoie les poids
#    dans leur échelle initiale.
# ------------------------------------------------------------

# ------------------------------------------------------------
# 4) Prédiction conditionnelle sur un élément du dataset
# ------------------------------------------------------------
def predict_conditional_path_data(
    path_data: dict[str, object],
    models: dict,
    split_final_by_barrier: bool = True) -> np.ndarray:
    """Applique les paniers conditionnels à une trajectoire.
    Si le produit est déjà rappelé, le cashflow répliqué est nul."""
    n_dates = len(path_data["dates"])
    prediction = np.zeros(n_dates, dtype=float)

    for i in range(n_dates):
        alive_before = bool(
            path_data["alive_before"][i]
        )

        if not alive_before:
            prediction[i] = 0.0
            continue

        is_final_date = (
            i == n_dates - 1
        )

        if (
            is_final_date
            and split_final_by_barrier
        ):
            state = bool(
                path_data["barrier_hit"][i]
            )
        else:
            state = None

        model_key = (i, state)

        # Si le sous-état de barrière contient trop peu de paths,
        # on utilise le modèle générique de la date.
        if model_key not in models:
            model_key = (i, None)

        model = models[model_key]
        feature_indices = model["feature_indices"]
        weights = model["weights"]

        X_date = path_data["X_curve"][
            i,
            feature_indices
        ]

        prediction[i] = float(
            X_date @ weights
        )

    return prediction


# ------------------------------------------------------------
# 5) Pipeline complet :
#    un panier par date, conditionnel à l'état du produit
# ------------------------------------------------------------
def run_conditional_benchmark_curve(
    product: AutocallProduct,
    spot0: float,
    valuation_date: str | pd.Timestamp,
    maturity_date: str | pd.Timestamp,
    rate: float,
    dividend_yield: float,
    vol: float,
    family: str = "full",
    n_paths: int = 2000,
    spot_min_mult: float = 0.70,
    spot_max_mult: float = 1.30,
    penalty: str = "l2",
    alpha: float = 1e-4,
    train_frac: float = 0.7,
    seed: int = 42,
    split_final_by_barrier: bool = True,
    min_state_paths: int = 20) -> dict[str, object]:
    """Apprend :
    - un panier par date parmi les paths encore vivants ;
    - deux paniers à maturité selon l'état de la barrière.

    Les poids sont appris uniquement sur les paths d'entraînement."""
    valuation_date = pd.Timestamp(valuation_date)
    maturity_date = pd.Timestamp(maturity_date)

    vanillas = build_conditional_vanilla_basis(
        product=product,
        spot0=spot0,
        valuation_date=valuation_date,
        maturity_date=maturity_date,
        family=family,
        spot_min_mult=spot_min_mult,
        spot_max_mult=spot_max_mult)

    dataset = build_conditional_curve_dataset(
        product=product,
        vanillas=vanillas,
        spot0=spot0,
        valuation_date=valuation_date,
        maturity_date=maturity_date,
        rate=rate,
        dividend_yield=dividend_yield,
        vol=vol,
        n_paths=n_paths,
        seed=seed)

    split = int(train_frac * n_paths)
    split = max(
        1,
        min(
            split,
            n_paths - 1 if n_paths > 1 else 1))

    train_set = dataset[:split]
    test_set = dataset[split:]

    dates = pd.DatetimeIndex(
        dataset[0]["dates"]
    )

    first_path = dataset[0]["path"]

    # Colonnes de vanilles associées à chaque date.
    feature_indices_by_date = {}

    for i, d in enumerate(dates):
        d = pd.Timestamp(d)

        feature_indices = []

        for j, vanilla in enumerate(vanillas):
            payment_date = effective_date_on_path(
                path=first_path,
                contractual_date=vanilla.maturity_date)

            if payment_date == d:
                feature_indices.append(j)

        if len(feature_indices) == 0:
            raise ValueError(
                f"Aucune vanille disponible à la date {d}."
            )

        feature_indices_by_date[i] = np.array(
            feature_indices,
            dtype=int)

    models = {}

    for i, d in enumerate(dates):
        feature_indices = (
            feature_indices_by_date[i]
        )

        alive_mask = np.array([
            bool(path_data["alive_before"][i])
            for path_data in train_set
        ])

        X_date = np.vstack([
            path_data["X_curve"][
                i,
                feature_indices
            ]
            for path_data in train_set
        ])

        y_date = np.array([
            path_data["y_curve"][i]
            for path_data in train_set
        ])

        # Modèle générique de la date, ajusté sur tous les paths vivants.
        X_alive = X_date[alive_mask]
        y_alive = y_date[alive_mask]

        if len(X_alive) == 0:
            raise ValueError(
                f"Aucun path vivant pour la date {d}."
            )

        weights = fit_linear(
            A=X_alive,
            b=y_alive,
            scale_features=True,
            penalty=penalty,
            alpha=alpha)

        models[(i, None)] = {
            "date": pd.Timestamp(d),
            "state": None,
            "feature_indices": feature_indices,
            "weights": weights,
            "n_train_paths": int(len(X_alive))}

        # À maturité, on peut distinguer les deux états de barrière.
        is_final_date = (
            i == len(dates) - 1
        )

        if (
            is_final_date
            and split_final_by_barrier
        ):
            for barrier_state in [False, True]:
                barrier_mask = np.array([
                    bool(path_data["barrier_hit"][i])
                    == barrier_state
                    for path_data in train_set
                ])

                state_mask = (
                    alive_mask
                    & barrier_mask
                )

                if (
                    int(np.sum(state_mask))
                    < min_state_paths
                ):
                    continue

                X_state = X_date[state_mask]
                y_state = y_date[state_mask]

                state_weights = fit_linear(
                    A=X_state,
                    b=y_state,
                    scale_features=True,
                    penalty=penalty,
                    alpha=alpha)

                models[(i, barrier_state)] = {
                    "date": pd.Timestamp(d),
                    "state": barrier_state,
                    "feature_indices": feature_indices,
                    "weights": state_weights,
                    "n_train_paths": int(len(X_state))}

    # Prédictions train
    pred_train_matrix = np.vstack([
        predict_conditional_path_data(
            path_data=path_data,
            models=models,
            split_final_by_barrier=split_final_by_barrier)
        for path_data in train_set
    ])

    y_train_matrix = np.vstack([
        path_data["y_curve"]
        for path_data in train_set
    ])

    # Prédictions test
    if len(test_set) > 0:
        pred_test_matrix = np.vstack([
            predict_conditional_path_data(
                path_data=path_data,
                models=models,
                split_final_by_barrier=split_final_by_barrier)
            for path_data in test_set
        ])

        y_test_matrix = np.vstack([
            path_data["y_curve"]
            for path_data in test_set
        ])
    else:
        pred_test_matrix = np.empty(
            (0, len(dates))
        )

        y_test_matrix = np.empty(
            (0, len(dates))
        )

    metrics_train = benchmark_metrics(
        y_train_matrix.reshape(-1),
        pred_train_matrix.reshape(-1))

    metrics_test = (
        benchmark_metrics(
            y_test_matrix.reshape(-1),
            pred_test_matrix.reshape(-1))
        if len(test_set) > 0
        else {}
    )

    # Métriques calculées uniquement sur les observations vivantes.
    train_alive_mask = np.vstack([
        path_data["alive_before"]
        for path_data in train_set
    ]).astype(bool)

    metrics_train_alive = benchmark_metrics(
        y_train_matrix[train_alive_mask],
        pred_train_matrix[train_alive_mask])

    if len(test_set) > 0:
        test_alive_mask = np.vstack([
            path_data["alive_before"]
            for path_data in test_set
        ]).astype(bool)

        metrics_test_alive = benchmark_metrics(
            y_test_matrix[test_alive_mask],
            pred_test_matrix[test_alive_mask])
    else:
        test_alive_mask = np.empty(
            (0, len(dates)),
            dtype=bool)

        metrics_test_alive = {}

    metrics_by_date = []

    if len(test_set) > 0:
        for i, d in enumerate(dates):
            y_date = y_test_matrix[:, i]
            pred_date = pred_test_matrix[:, i]
            alive_date = test_alive_mask[:, i]

            metrics_all = benchmark_metrics(
                y_date,
                pred_date)

            row = {
                "date": pd.Timestamp(d),
                "n_test": int(len(y_date)),
                "n_alive": int(np.sum(alive_date)),
                "mae": metrics_all["mae"],
                "rmse": metrics_all["rmse"],
                "max_abs_error": metrics_all["max_abs_error"],
                "mean_error": metrics_all["mean_error"]}

            if np.any(alive_date):
                metrics_alive = benchmark_metrics(
                    y_date[alive_date],
                    pred_date[alive_date])

                row.update({
                    "mae_alive": metrics_alive["mae"],
                    "rmse_alive": metrics_alive["rmse"],
                    "max_abs_error_alive": metrics_alive["max_abs_error"],
                    "mean_error_alive": metrics_alive["mean_error"]})

            metrics_by_date.append(row)

    metrics_by_date = pd.DataFrame(
        metrics_by_date
    )

    # Tableau lisible de tous les paniers conditionnels.
    weights_rows = []

    for (date_index, state), model in models.items():
        feature_indices = model["feature_indices"]

        if state is None:
            state_name = "alive"
        elif state:
            state_name = "barrier_hit"
        else:
            state_name = "barrier_not_hit"

        for local_index, feature_index in enumerate(feature_indices):
            vanilla = vanillas[feature_index]

            weights_rows.append({
                "date": model["date"],
                "state": state_name,
                "n_train_paths": model["n_train_paths"],
                "name": vanilla.name,
                "kind": vanilla.__class__.__name__,
                "maturity_date": pd.Timestamp(
                    vanilla.maturity_date),
                "strike": float(vanilla.strike),
                "weight": float(
                    model["weights"][local_index])})

    weights = pd.DataFrame(
        weights_rows
    )

    weights["absolute_weight"] = (
        weights["weight"].abs()
    )

    weights = weights.sort_values(
        ["date", "state", "absolute_weight"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return {
        "vanillas": vanillas,
        "dates": dates,
        "dataset": dataset,
        "train_set": train_set,
        "test_set": test_set,
        "models": models,
        "weights": weights,
        "pred_train_matrix": pred_train_matrix,
        "pred_test_matrix": pred_test_matrix,
        "y_train_matrix": y_train_matrix,
        "y_test_matrix": y_test_matrix,
        "metrics_train": metrics_train,
        "metrics_test": metrics_test,
        "metrics_train_alive": metrics_train_alive,
        "metrics_test_alive": metrics_test_alive,
        "metrics_by_date": metrics_by_date,
        "family": family,
        "split": split,
        "split_final_by_barrier": split_final_by_barrier}


# ------------------------------------------------------------
# 6) Construction des features sur un nouveau path
# ------------------------------------------------------------
def conditional_features_on_path(
    vanillas: list[VanillaProduct],
    path: pd.Series,
    valuation_date: str | pd.Timestamp,
    rate: float,
    curve_dates: pd.DatetimeIndex) -> np.ndarray:
    """Construit la matrice de payoffs des vanilles
    pour une trajectoire donnée."""
    valuation_date = pd.Timestamp(valuation_date)
    curve_dates = pd.DatetimeIndex(curve_dates)

    path = path.sort_index().dropna()
    path = path.loc[
        path.index >= valuation_date
    ]

    X_curve = np.zeros(
        (len(curve_dates), len(vanillas)),
        dtype=float)

    date_to_index = {
        pd.Timestamp(d): i
        for i, d in enumerate(curve_dates)
    }

    for j, vanilla in enumerate(vanillas):
        payment_date = effective_date_on_path(
            path=path,
            contractual_date=vanilla.maturity_date)

        if payment_date not in date_to_index:
            raise ValueError(
                f"La maturité {payment_date} de {vanilla.name} "
                "ne correspond à aucune date de la courbe."
            )

        i = date_to_index[payment_date]

        X_curve[i, j] = (
            vanilla_discounted_payoff_from_path(
                vanilla=vanilla,
                path=path,
                valuation_date=valuation_date,
                rate=rate)
        )

    return X_curve


# ------------------------------------------------------------
# 7) Application des paniers conditionnels sur un nouveau path
# ------------------------------------------------------------
def conditional_portfolio_value_curve_on_path(
    result,
    product: AutocallProduct,
    path: pd.Series,
    valuation_date: str | pd.Timestamp,
    rate: float):
    """Applique la stratégie conditionnelle sur un nouveau path.
    Retourne :
    - les cashflows de l'autocall ;
    - les cashflows répliqués ;
    - les états du produit."""
    curve_dates = pd.DatetimeIndex(
        result["dates"]
    )

    target_curve = autocall_value_curve_on_path(
        product=product,
        path=path,
        valuation_date=valuation_date,
        rate=rate,
        curve_dates=curve_dates)

    X_curve = conditional_features_on_path(
        vanillas=result["vanillas"],
        path=path,
        valuation_date=valuation_date,
        rate=rate,
        curve_dates=curve_dates)

    state_curve = autocall_state_curve_on_path(
        product=product,
        path=path,
        valuation_date=valuation_date,
        curve_dates=curve_dates)

    path_data = {
        "path": path,
        "dates": curve_dates,
        "X_curve": X_curve,
        "y_curve": target_curve.to_numpy(dtype=float),
        "alive_before": state_curve[
            "alive_before"
        ].to_numpy(dtype=bool),
        "barrier_hit": state_curve[
            "barrier_hit"
        ].to_numpy(dtype=bool)}

    replica_values = predict_conditional_path_data(
        path_data=path_data,
        models=result["models"],
        split_final_by_barrier=result[
            "split_final_by_barrier"])

    replica_curve = pd.Series(
        replica_values,
        index=curve_dates,
        name="conditional_replica_cashflow")

    return (
        target_curve,
        replica_curve,
        state_curve)

In [ ]:
# ============================================================
# EXÉCUTION DU BENCHMARK CONDITIONNEL
# ============================================================

# Path réellement hors échantillon.
test_path = simulate_gbm_path(spot0=spot0,start_date=pd.Timestamp(valuation_date),end_date=pd.Timestamp(maturity_date),rate=rate,dividend_yield=dividend_yield,vol=vol,seed=95020)

for penalty in ["l2", "l1", "elastic_net"]:
    for family in ["calls", "calls_puts", "full"]:
    
        res_conditional = run_conditional_benchmark_curve(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family=family,n_paths=2000,penalty=penalty,alpha=1e-4,train_frac=0.7,seed=42,split_final_by_barrier=True,min_state_paths=20)

        (target_conditional_curve,replica_conditional_curve,state_conditional_curve) = conditional_portfolio_value_curve_on_path(result=res_conditional,product=prod,path=test_path,valuation_date=valuation_date,rate=rate)

        plot_timewise_curve_benchmark(target_conditional_curve.index,target_conditional_curve.to_numpy(),replica_conditional_curve.to_numpy(),title_prefix=f"[Conditionnel / {family} / {penalty}] ")

        print("Métriques globales train :")
        display(pd.Series(res_conditional["metrics_train"]))

        print("Métriques globales test :")
        display(pd.Series(res_conditional["metrics_test"]))

        print("Métriques train sur les paths vivants :")
        display(pd.Series(res_conditional["metrics_train_alive"]))

        print("Métriques test sur les paths vivants :")
        display(pd.Series(res_conditional["metrics_test_alive"]))

        print("Métriques test par date :")
        display(res_conditional["metrics_by_date"])

        print("États du path affiché :")
        display(state_conditional_curve)

        print("Paniers conditionnels :")
        display(res_conditional["weights"].head(50))

In [ ]:
# ============================================================
# EXÉCUTION DU BENCHMARK CONDITIONNEL
# ============================================================

# Path réellement hors échantillon.
test_path = simulate_gbm_path(spot0=spot0,start_date=pd.Timestamp(valuation_date),end_date=pd.Timestamp(maturity_date),rate=rate,dividend_yield=dividend_yield,vol=vol,seed=95020)

for penalty in ["l2", "l1", "elastic_net"]:
    for family in ["full"]:
    
        res_conditional = run_conditional_benchmark_curve(product=prod,spot0=spot0,valuation_date=valuation_date,maturity_date=maturity_date,rate=rate,dividend_yield=dividend_yield,vol=vol,family=family,n_paths=2000,penalty=penalty,alpha=1e-4,train_frac=0.7,seed=42,split_final_by_barrier=False,min_state_paths=20)

        (target_conditional_curve,replica_conditional_curve,state_conditional_curve) = conditional_portfolio_value_curve_on_path(result=res_conditional,product=prod,path=test_path,valuation_date=valuation_date,rate=rate)

        plot_timewise_curve_benchmark(target_conditional_curve.index,target_conditional_curve.to_numpy(),replica_conditional_curve.to_numpy(),title_prefix=f"[Conditionnel / {family} / {penalty}] ")

        print("Métriques globales train :")
        display(pd.Series(res_conditional["metrics_train"]))

        print("Métriques globales test :")
        display(pd.Series(res_conditional["metrics_test"]))

        print("Métriques train sur les paths vivants :")
        display(pd.Series(res_conditional["metrics_train_alive"]))

        print("Métriques test sur les paths vivants :")
        display(pd.Series(res_conditional["metrics_test_alive"]))

        print("Métriques test par date :")
        display(res_conditional["metrics_by_date"])

        print("États du path affiché :")
        display(state_conditional_curve)

        print("Paniers conditionnels :")
        display(res_conditional["weights"].head(50))

## 7. Grille d’entraînement

Le fit ne doit pas être fait uniquement au spot initial.  
C’est une erreur fréquente et trompeuse.

On entraîne le portefeuille sur une grille de points comprenant :

- plusieurs spots ;
- plusieurs temps résiduels ;
- plusieurs états du produit ;
- éventuellement plusieurs scénarios de volatilité ou de skew.

Cela permet de mesurer la qualité du fit non seulement localement, mais aussi en robustesse.

---

## 8. Régularisation

Le problème de fit est souvent mal conditionné.  
Il faut donc régulariser.

On peut utiliser par exemple :

$$
\min_w \|Aw - b\|^2 + \lambda \|w\|^2
$$

avec éventuellement :

- une pénalité de courbure sur les poids ;
- des contraintes de signe ;
- des contraintes de budget ;
- des contraintes de neutralité delta.

La régularisation n’est pas un détail technique : elle est essentielle pour obtenir un portefeuille interprétable et stable.

---

## 9. Critères d’évaluation

La première partie du notebook doit mesurer au moins :

- l’erreur de prix moyenne ;
- l’erreur maximale ;
- la stabilité des poids ;
- la sensibilité du fit au changement de spot ;
- la sensibilité du fit à la grille de calibration ;
- le comportement hors échantillon.

Un bon benchmark naïf n’est pas celui qui fit le mieux un seul point, mais celui qui se comporte de manière acceptable dans des conditions variées.

---

## 10. Lecture de cette baseline

Cette baseline doit être présentée comme un point de comparaison volontairement simple.

Elle ne cherche pas à exploiter toute la structure du produit.  
Elle cherche plutôt à montrer ce que permet, et surtout ce que ne permet pas, un fit direct par panier fini de vanilles.


## 11. Remarques pratiques

Les cellules suivantes construisent une petite partie “marché” du notebook. Si `yfinance` n’est pas installé, cette section s’arrête proprement sans faire planter le notebook.


## 12. Données de marché optionnelles

Cette étape est utile pour récupérer un historique de sous-jacent et une chaîne d’options de test. Elle n’est pas indispensable pour comprendre la logique de réplication, mais elle rend l’exemple plus concret.


## 13. Vérification sur un sous-jacent de marché

On prend ici un ticker de test pour illustrer la construction de l’univers de vanilles à partir de données observées.


## 14. Lecture de la chaîne d’options

L’objectif est de garder uniquement les colonnes utiles pour la suite : strike, bid, ask, volatilité implicite, volume et open interest.


## 15. Contrôle de robustesse

Cette partie sert surtout à vérifier que les données récupérées sont cohérentes avant de les utiliser dans une calibration.


## 16. Préparation du panier de vanilles

On regroupe les calls et les puts dans un seul tableau pour faciliter la construction d’un univers de réplication.


## 17. Nettoyage des colonnes

On garde volontairement peu de colonnes au départ pour éviter d’introduire du bruit inutile dans le notebook.


## 18. Dernier affichage avant calibration

Ce tableau résume l’univers de départ utilisé pour la suite des tests.


In [ ]:
ticker = "SPY"   # ou "AAPL"

if yf is None:      print("yfinance n'est pas installé dans cet environnement : la partie données de marché est ignorée.")
else:
    tk = yf.Ticker(ticker)

    # 1) Historique du sous-jacent
    spot_hist = yf.download(ticker,start="2025-01-01",end="2026-06-30",interval="1d",auto_adjust=True,progress=False)
    print("Historique spot :")
    display(spot_hist.tail())
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,gridspec_kw={"height_ratios": [3, 1]})
    ax1.plot(spot_hist.index, spot_hist["Close"], label="Close")
    ax1.plot(spot_hist.index, spot_hist["High"], label="High")
    ax1.plot(spot_hist.index, spot_hist["Low"], label="Low")
    ax1.set_title(f"Historique du sous-jacent ({ticker})")
    ax1.set_ylabel("Prix")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(spot_hist.index, spot_hist["Volume"], color="gray")
    ax2.set_ylabel("Volume")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    spot = spot_hist["Close"].dropna().iloc[-1].item()  # dernier cours de clôture disponible, utilisé comme spot de référence 
    valuation_date = pd.Timestamp(spot_hist.index[-1])

    # 2) Liste des maturités disponibles
    expiries = tk.options
    if not expiries:        print("Aucune maturité disponible pour ce ticker.")
    else:
        print("Maturities disponibles :", expiries[:])
        print("Nombre de maturités :", len(expiries))
        expiry = expiries[0]
        maturity = pd.Timestamp(expiry)

        # 3) Chaîne d'options
        chain = tk.option_chain(expiry)
        calls = chain.calls.copy()
        puts = chain.puts.copy()

        # 4) Construction d'un petit univers de vanilles
        calls["kind"] = "call"
        puts["kind"] = "put"
        vanillas_market = pd.concat([calls, puts], ignore_index=True)

        cols = [c for c in ["contractSymbol","kind","strike","lastPrice","impliedVolatility","expiration"] if c in vanillas_market.columns]
        vanillas_market = vanillas_market[cols].copy()
        
        
        # Prix Black-Scholes théorique ligne par ligne
        def _bs_price_row(row):
            sigma = row["impliedVolatility"]
            if pd.isna(sigma) or sigma <= 0:    return np.nan

            if row["kind"] == "call":   prod = EuropeanCall(name="tmp",strike=float(row["strike"]),maturity_date=maturity,notional=1.0)
            else:                       prod = EuropeanPut(name="tmp",strike=float(row["strike"]),maturity_date=maturity,notional=1.0)

            return prod.price_bs(spot, valuation_date, rate, dividend_yield, sigma)

        vanillas_market["Benchmark price BS"] = vanillas_market.apply(_bs_price_row, axis=1)
        display(vanillas_market.head(10))
